**Prompt:** Máme 8-bitová obrazová data, která obsahují maximálně 14 hodnot intenzit včetně pozadí s nulovou intenzitou. Hodnoty objektů mají intenzity 1-13. Každý obraz může obsahovat více objektů se stejnou intenzitou. Potřebuji každý obraz přepočítat tak, aby objekty s původní intenzitou 1 začínaly v novém obraze s intenzitou 1001. Pokud bude objektů s intenzitou 1 v původním obraze více, každý by měl mít svou vlastní intenzitu o 1 vyšší. Tj., první objekt s intenzitou 1 bude mít 1001, druhý nalezený objekt s intenzitou 1 bude mít intenzitu 1002, třetí pak 1003 atd. První objekt s intenzitou pak 2 bude mít 2001, druhý nalezený objekt s intenzitou 2 bude mít intenzitu 2002, třetí pak 2003 atd. Totéž platí pro všechny původní intenzity (1-13) a objekty s těmito intenzitami v obraze.

In [32]:
import os
import numpy as np
from scipy.ndimage import label
from PIL import Image, UnidentifiedImageError
from typing import Dict

In [33]:
cesta_k_adresari = r'd:\Programovani\_Empanada_Training\250616 Data_for_training_Mito_Kristy\Kristy_All\10_images_masks_For_Panoptic_Seg\masks'
cilovy_adresar = r'd:\Programovani\_Empanada_Training\250616 Data_for_training_Mito_Kristy\Kristy_All\10_images_masks_For_Panoptic_Seg\masks_recomputed_for_panoptic_segmentation'

In [34]:
def nacti_tif_do_slovniku(adresar_cesta: str) -> Dict[str, np.ndarray]:
    """
    Načte všechny soubory .tif z daného adresáře a vrátí je jako slovník.

    Klíčem ve slovníku je název souboru a hodnotou je NumPy pole s daty obrazu.

    Args:
        adresar_cesta (str): Cesta k adresáři se soubory .tif.

    Returns:
        Dict[str, np.ndarray]: Slovník, kde {název_souboru: obraz_jako_pole}.
                               Pokud adresář neexistuje nebo je prázdný,
                               vrátí prázdný slovník.
    """
    slovnik_obrazu = {}
    
    # Ověření, zda zadaná cesta existuje a je to adresář
    if not os.path.isdir(adresar_cesta):
        print(f"❌ Chyba: Adresář na cestě '{adresar_cesta}' nebyl nalezen.")
        return slovnik_obrazu

    print(f"🔎 Prohledávám adresář: '{adresar_cesta}'...")

    # Získání seřazeného seznamu souborů pro konzistentní pořadí
    soubory_v_adresari = sorted(os.listdir(adresar_cesta))

    # Iterace přes všechny soubory v adresáři
    for nazev_souboru in soubory_v_adresari:
        # Kontrola přípony souboru (insensitivní na velikost písmen)
        if nazev_souboru.lower().endswith(('.tif', '.tiff')):
            plna_cesta_k_souboru = os.path.join(adresar_cesta, nazev_souboru)
            try:
                # Otevření obrázku a jeho konverze na NumPy pole
                with Image.open(plna_cesta_k_souboru) as img:
                    obraz_pole = np.array(img)
                    # Uložení do slovníku: klíč = název souboru, hodnota = pole
                    slovnik_obrazu[nazev_souboru] = obraz_pole
                    print(f"✔️ Načten soubor: '{nazev_souboru}' (rozměry: {obraz_pole.shape})")
            
            except UnidentifiedImageError:
                print(f"⚠️ Varování: Soubor '{nazev_souboru}' má příponu .tif, ale není to platný obrazový formát.")
            except Exception as e:
                print(f"❌ Chyba při zpracování souboru '{nazev_souboru}': {e}")

    if not slovnik_obrazu:
        print("ℹ️ V adresáři nebyly nalezeny žádné platné .tif/.tiff soubory.")
        
    return slovnik_obrazu


In [35]:
nactene_obrazy_dict = nacti_tif_do_slovniku(cesta_k_adresari)

if nactene_obrazy_dict:
    print(f"\n✅ Úspěšně načteno {len(nactene_obrazy_dict)} obrazových souborů do slovníku.")

🔎 Prohledávám adresář: 'd:\Programovani\_Empanada_Training\250616 Data_for_training_Mito_Kristy\Kristy_All\10_images_masks_For_Panoptic_Seg\masks'...
✔️ Načten soubor: 'Ctrl_Robin_20000x_mito.tif' (rozměry: (2044, 2042))
✔️ Načten soubor: 'Ctrl_Robin_25000x2_mito.tif' (rozměry: (2041, 2044))
✔️ Načten soubor: 'Ctrl_Robin_30000x_mito.tif' (rozměry: (2043, 1981))
✔️ Načten soubor: 'Ctrl_Robin_40000x4_mito.tif' (rozměry: (2048, 1999))
✔️ Načten soubor: 'Ctrl_Robin_40000x5_mito.tif' (rozměry: (2022, 2041))
✔️ Načten soubor: 'Ctrl_Robin_40000x_mito.tif' (rozměry: (2024, 2028))
✔️ Načten soubor: 'Ctrl_Robin_60000x1_mito.tif' (rozměry: (2022, 2028))
✔️ Načten soubor: 'Ctrl_Robin_60000x3_mito.tif' (rozměry: (2003, 2035))
✔️ Načten soubor: 'Ctrl_Robin_60000x_mito.tif' (rozměry: (2033, 2047))
✔️ Načten soubor: 'Cyril_kontrola_002_30000x_mito.tif' (rozměry: (2047, 2047))
✔️ Načten soubor: 'Cyril_kontrola_006_20000x_mito.tif' (rozměry: (2043, 2038))
✔️ Načten soubor: 'Cyril_kontrola_011_30000x_mit

In [36]:
def prepocti_intenzity_objektu(puvodni_obraz: np.ndarray) -> np.ndarray:
    """
    Přepočítá intenzity objektů v obraze podle zadaných pravidel.

    Objekty s původní intenzitou `i` budou mít v novém obraze intenzity
    začínající na `i * 1000 + 1`. Každý další samostatný objekt se stejnou
    původní intenzitou `i` bude mít hodnotu o 1 vyšší.

    Args:
        puvodni_obraz (np.ndarray): Vstupní 2D pole (obraz) s 8-bitovými daty.

    Returns:
        np.ndarray: Nový 2D pole (obraz) s přepočítanými intenzitami.
                    Datový typ bude 'int16', aby se předešlo přetečení.
    """
    if not isinstance(puvodni_obraz, np.ndarray):
        raise TypeError("Vstupní obraz musí být typu numpy.ndarray")

    # Vytvoříme nový obraz se stejnými rozměry a datovým typem,
    # který pojme i vysoké hodnoty nových intenzit (např. 13000+).
    novy_obraz = np.zeros(puvodni_obraz.shape, dtype=np.int16)

    # Projdeme všechny možné původní intenzity objektů (1 až 13)
    for orig_intenzita in range(1, 14):
        # Vytvoříme binární masku, kde jsou `True` pouze pixely
        # s aktuálně zpracovávanou intenzitou.
        maska = (puvodni_obraz == orig_intenzita)

        # Pokud v obraze nejsou žádné pixely s touto intenzitou, pokračujeme dál.
        if not np.any(maska):
            continue

        # Funkce `label` najde všechny souvislé oblasti (objekty) v masce.
        # `oznacene_objekty` je pole, kde má každý objekt své unikátní číslo (1, 2, 3...).
        # `pocet_objektu` je celkový počet nalezených objektů pro danou intenzitu.
        oznacene_objekty, pocet_objektu = label(maska)

        # Pokud nebyly nalezeny žádné souvislé objekty, přeskočíme.
        if pocet_objektu == 0:
            continue
            
        # Projdeme každý nalezený objekt (od 1 do `pocet_objektu`)
        for i in range(1, pocet_objektu + 1):
            # Vypočítáme novou intenzitu podle vzorce.
            nova_intenzita = orig_intenzita * 1000 + i
            
            # Najdeme v `novy_obraz` pixely, které odpovídají aktuálnímu
            # objektu (s označením `i`) a přiřadíme jim novou intenzitu.
            novy_obraz[oznacene_objekty == i] = nova_intenzita

    return novy_obraz

In [3]:

# --- Příklad použití ---

# Vytvoření vzorového 8-bitového obrazu (15x15 pixelů)
# Pozadí je 0.
# Intenzita 1: Dva oddělené objekty.
# Intenzita 2: Jeden velký objekt.
# Intenzita 7: Tři malé, oddělené objekty (jednopixelové).
# Intenzita 13: Jeden objekt.
puvodni_obraz_data = np.array([
    [0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 1, 1, 0, 2, 2, 2, 0, 0, 0, 0, 7, 0, 0, 0],
    [0, 0, 0, 0, 2, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 2, 2, 2, 0, 1, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 13, 13, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 7, 0, 0, 13, 13, 0, 0, 0, 0, 0, 0, 0],
], dtype=np.uint8)

print("--- Původní obraz: ---")
print(puvodni_obraz_data)

# Zavolání funkce pro přepočet
novy_prepocteny_obraz = prepocti_intenzity_objektu(puvodni_obraz_data)

print("\n--- Nový, přepočtený obraz: ---")
print(novy_prepocteny_obraz)

--- Původní obraz: ---
[[ 0  1  1  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  1  1  0  2  2  2  0  0  0  0  7  0  0  0]
 [ 0  0  0  0  2  2  2  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  2  2  2  0  1  1  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  1  1  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  7  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0 13 13  0  0  0  0  0  0  0]
 [ 0  0  0  7  0  0 13 13  0  0  0  0  0  0  0]]

--- Nový, přepočtený obraz: ---
[[    0  1001  1001     0     0     0     0     0     0     0     0     0
      0     0     0]
 [    0  1001  1001     0  2001  2001  2001     0     0     0     0  7001
      0     0     0]
 [    0     0     0     0  2001  2001  2001     0     0     0     0     0
      0     0     0]
 [    0     0     0     0  2001  2001  2001     0  1002  1002     0     0
      0     0     0]
 [    0     0     0     0     0     0     0     0  1002  1002     0     0
      0     0     0]
 [    0     0     0     0     0     

In [37]:
def uloz_16bit_obrazy_pil(snimky: Dict[str, np.ndarray], cilovy_adresar: str):
    os.makedirs(cilovy_adresar, exist_ok=True)

    for nazev, obraz in snimky.items():
        # Bezpečný název souboru
        cesta = os.path.join(cilovy_adresar, nazev)

        # Labels recomputed by Martin
        obrazNew = prepocti_intenzity_objektu(obraz)

        # Grayscale 16bit
        if obrazNew.ndim == 2:
            img = Image.fromarray(obrazNew, mode='I;16')
        else:
            raise ValueError(f"Obraz '{nazev}' má nepodporovaný formát: shape {obrazNew.shape}")

        img.save(cesta, format='TIFF')
        print(f"Uložen obraz: {cesta}")


In [38]:
uloz_16bit_obrazy_pil(nactene_obrazy_dict, cilovy_adresar)

Uložen obraz: d:\Programovani\_Empanada_Training\250616 Data_for_training_Mito_Kristy\Kristy_All\10_images_masks_For_Panoptic_Seg\masks_recomputed_for_panoptic_segmentation\Ctrl_Robin_20000x_mito.tif
Uložen obraz: d:\Programovani\_Empanada_Training\250616 Data_for_training_Mito_Kristy\Kristy_All\10_images_masks_For_Panoptic_Seg\masks_recomputed_for_panoptic_segmentation\Ctrl_Robin_25000x2_mito.tif
Uložen obraz: d:\Programovani\_Empanada_Training\250616 Data_for_training_Mito_Kristy\Kristy_All\10_images_masks_For_Panoptic_Seg\masks_recomputed_for_panoptic_segmentation\Ctrl_Robin_30000x_mito.tif
Uložen obraz: d:\Programovani\_Empanada_Training\250616 Data_for_training_Mito_Kristy\Kristy_All\10_images_masks_For_Panoptic_Seg\masks_recomputed_for_panoptic_segmentation\Ctrl_Robin_40000x4_mito.tif
Uložen obraz: d:\Programovani\_Empanada_Training\250616 Data_for_training_Mito_Kristy\Kristy_All\10_images_masks_For_Panoptic_Seg\masks_recomputed_for_panoptic_segmentation\Ctrl_Robin_40000x5_mito.t